In [ ]:
# own mcp server
# client for it 

# tools, resources

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

In [ ]:
from accounts import Account

In [ ]:
account = Account.get("Divya")
account

In [ ]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

In [ ]:
account.report()

In [ ]:
account.list_transactions()

In [ ]:
# our own mcp server
# accounts.py as mcp server -> accounts_server.py

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [ ]:
mcp_tools

In [ ]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Divya and my account is under the name Divya. What's my balance and my holdings?"
model = "gpt-4.1-mini"

In [ ]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


In [ ]:
# we've also exposed resources from our mcp, but the above code can only fetch tools
# in order to access resources, u need to write a client

# openai released a new version, wehre u can pass the mcp server and you don't need to create tools in openai format
# this is the way, how it was done earlier and if you don't want resources, u can use above code
# earlier openai didn't support using mcp servers, for resources you still need to create a client and use it in old way

from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

In [ ]:
# openai tools here are a wrapper around mcp tools which created the client whcih called the server

request = "My name is Divya and my account is under the name Divya. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

In [ ]:
# reading resources doesn't come with openai sdk packages, this is the way to do it

context = await read_accounts_resource("divya")
print(context)

In [ ]:
# directly calling the business logic instead of calling it from tool and from server, the above calls the same code but with mcp

from accounts import Account
Account.get("divya").report()